# IduEdu paper — accessibility and scenario figures

Figures for B5/B6. The notebook only reads cached artifacts from `results/accessibility`; it never rebuilds graphs or reruns routing.

In [ ]:
from pathlib import Path
import json
import re

import geopandas as gpd
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap
from matplotlib.patches import Patch
from shapely.geometry import LineString, MultiLineString, MultiPolygon, Polygon
from shapely.ops import polygonize

RESULTS = Path('../results/accessibility')
FIGURES = Path('.')
DPI = 300
VISUAL_BOUNDS = None
STROKE = [pe.Stroke(linewidth=2.0, foreground='white'), pe.Normal()]
C = {
    'iduedu': '#00A8FF',
    'walk': '#FF9F0A',
    'intermodal': '#00A8FF',
    'pt': '#AF52DE',
    'green': '#32D74B',
    'red': '#FF375F',
    'gray': '#8E8E93',
}
GAIN_COLORS = {
    'no_effect': '#B8B8B8',
    'moderate': '#FF9F0A',
    'substantial': '#00A8FF',
    'newly_reachable': '#32D74B',
    'unreachable': '#FF375F',
}
PT_COLORS = {
    'bus': '#AF52DE',
    'tram': '#FF375F',
    'trolleybus': '#32D74B',
    'subway': '#007AFF',
    'train': '#5E5CE6',
}

def style_ax(ax, grid=False):
    ax.set_facecolor('white')
    for spine in ax.spines.values():
        spine.set_visible(False)
    if grid:
        ax.grid(True, color='black', alpha=0.10, linewidth=0.8)
    ax.tick_params(axis='both', colors='black', labelsize=9)

def style_map(ax):
    ax.set_axis_off()
    ax.set_aspect('equal')
    if VISUAL_BOUNDS is not None:
        minx, miny, maxx, maxy = VISUAL_BOUNDS
        ax.set_xlim(minx, maxx)
        ax.set_ylim(miny, maxy)

def save(fig, name, *, tight=True, crop=True):
    fig.patch.set_facecolor('white')
    if tight:
        fig.tight_layout()
    path = FIGURES / name
    fig.savefig(path, dpi=DPI, bbox_inches='tight' if crop else None, facecolor='white')
    print(f'saved {path}')

def load_gdf(name):
    path = RESULTS / name
    if not path.exists():
        print(f'[skip] missing {path}')
        return None
    return gpd.read_parquet(path)

def plot_network(
    ax, walk_edges=None, pt_edges=None, walk_color='#D8D8D8',
    pt_alpha=0.75, pt_linewidth=1.0, pt_color=None,
):
    if visual_boundary_m is not None:
        visual_boundary_m.boundary.plot(ax=ax, color='#4A4A4A', linewidth=0.8, zorder=0)
    if walk_edges is not None and not walk_edges.empty:
        walk_edges.plot(ax=ax, color=walk_color, linewidth=0.25, alpha=0.45, zorder=1, rasterized=True)
    if pt_edges is not None and not pt_edges.empty:
        for mode, group in pt_edges.groupby('type'):
            group.plot(
                ax=ax, color=pt_color or PT_COLORS.get(str(mode), C['pt']),
                linewidth=pt_linewidth, alpha=pt_alpha, zorder=2, rasterized=True,
            )

def plot_highlighted_routes(ax, route_edges):
    if route_edges is None or route_edges.empty:
        return
    route_edges.plot(ax=ax, color='white', linewidth=2.4, alpha=0.3, zorder=12)
    route_edges.plot(ax=ax, color='#FF1744', linewidth=.8, alpha=0.7, zorder=13)

def projected_hexbin(ax, points, values, *, gridsize, **kwargs):
    if points.crs is None or not points.crs.is_projected:
        raise ValueError('Hexbin input must use a projected metric CRS')
    minx, miny, maxx, maxy = VISUAL_BOUNDS or tuple(points.total_bounds)
    nx = int(gridsize)
    ny = max(1, round((maxy - miny) * nx / (np.sqrt(3) * (maxx - minx))))
    return ax.hexbin(
        points.geometry.x, points.geometry.y, C=np.asarray(values, dtype=float),
        gridsize=(nx, ny), extent=(minx, maxx, miny, maxy), **kwargs,
    )

def zoom_to_layers(ax, layers, *, padding_fraction=0.12, min_padding=700.0):
    bounds = [layer.total_bounds for layer in layers if layer is not None and not layer.empty]
    if not bounds:
        return
    bounds = np.asarray(bounds)
    minx, miny = bounds[:, :2].min(axis=0)
    maxx, maxy = bounds[:, 2:].max(axis=0)
    padding = max(min_padding, max(maxx - minx, maxy - miny) * padding_fraction)
    ax.set_xlim(minx - padding, maxx + padding)
    ax.set_ylim(miny - padding, maxy + padding)

def polygons_to_multilinestring(geom: Polygon | MultiPolygon) -> MultiLineString:
    def polygon_to_lines(polygon: Polygon) -> list[LineString]:
        return [LineString(polygon.exterior), *(LineString(ring) for ring in polygon.interiors)]

    if geom.geom_type == 'Polygon':
        return MultiLineString(polygon_to_lines(geom))
    if geom.geom_type == 'MultiPolygon':
        return MultiLineString([line for polygon in geom.geoms for line in polygon_to_lines(polygon)])
    raise TypeError(f'Expected Polygon or MultiPolygon, got {geom.geom_type}')

def combine_geometry(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    crs = gdf.crs
    boundaries = gdf['geometry'].apply(polygons_to_multilinestring).union_all()
    enclosures = gpd.GeoDataFrame(geometry=list(polygonize(boundaries)), crs=crs)
    enclosure_points = enclosures.copy()
    enclosure_points.geometry = enclosures.representative_point()
    joined = gpd.sjoin(enclosure_points, gdf, how='inner', predicate='within').reset_index()
    columns = [column for column in joined.columns if column not in {'index', 'geometry'}]
    joined = joined.groupby('index').agg({column: list for column in columns})
    joined['geometry'] = enclosures.geometry
    return gpd.GeoDataFrame(joined, geometry='geometry', crs=crs)

def scenario_slug(value):
    text = re.sub(r'[^0-9A-Za-zА-Яа-я_-]+', '_', str(value)).strip('_')
    return text or 'unnamed'

origins = load_gdf('origins_accessibility.parquet')
schools = load_gdf('schools.parquet')
walk_edges = load_gdf('map_walk_edges.parquet')
pt_edges = load_gdf('map_pt_edges.parquet')
visual_boundary = load_gdf('visual_boundary_421007.parquet')
summary = pd.read_csv(RESULTS / 'accessibility_summary.csv') if (RESULTS / 'accessibility_summary.csv').exists() else None

visual_boundary_m = None
if origins is not None:
    MAP_CRS = visual_boundary.estimate_utm_crs() if visual_boundary is not None else origins.estimate_utm_crs()
    origins_m = origins.to_crs(MAP_CRS)
    origins_points = origins_m.copy()
    origins_points.geometry = origins_points.geometry.representative_point()
    schools_m = schools.to_crs(MAP_CRS) if schools is not None else None
    walk_edges_m = walk_edges.to_crs(MAP_CRS) if walk_edges is not None else None
    pt_edges_m = pt_edges.to_crs(MAP_CRS) if pt_edges is not None else None
    if visual_boundary is not None:
        visual_boundary_m = visual_boundary.to_crs(MAP_CRS)
        boundary_geometry = visual_boundary_m.geometry.union_all()
        VISUAL_BOUNDS = tuple(visual_boundary_m.total_bounds)
        origins_points = origins_points.loc[origins_points.intersects(boundary_geometry)].copy()
        if schools_m is not None:
            schools_m = schools_m.loc[schools_m.intersects(boundary_geometry)].copy()
        if walk_edges_m is not None:
            walk_edges_m = walk_edges_m.clip(visual_boundary_m)
        if pt_edges_m is not None:
            pt_edges_m = pt_edges_m.clip(visual_boundary_m)
    CITY_SCALE = len(origins) > 5_000
    ORIGIN_MARKER_SIZE = 1.2 if CITY_SCALE else 18
    SCHOOL_MARKER_SIZE = 7 if CITY_SCALE else 75
    SCENARIO_MARKER_SIZE = 1.4 if CITY_SCALE else 21

## Accessibility metrics

In [ ]:
if origins is not None and summary is not None:
    opportunity_gain = origins.opportunity_gain_30.clip(lower=0)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.3), dpi=DPI)

    gain_labels = ['No gain', '1–5', '6–20', '21–50', '>50']
    gain_groups = pd.cut(
        opportunity_gain, bins=[-0.5, 0.5, 5.5, 20.5, 50.5, np.inf], labels=gain_labels
    )
    gain_shares = gain_groups.value_counts(normalize=True).reindex(gain_labels).fillna(0) * 100
    gain_bars = axes[0].barh(gain_labels, gain_shares, color=C['iduedu'], edgecolor='white')
    axes[0].bar_label(gain_bars, labels=[f'{value:.1f}%' for value in gain_shares], padding=3)
    axes[0].invert_yaxis()
    axes[0].set_title('30-min gain by residential origin', fontweight='bold')
    axes[0].set_xlabel('Origins (%)')
    axes[0].set_ylabel('Additional reachable schools')
    style_ax(axes[0], grid=True)

    thresholds = [15, 30, 45, 60]
    x = np.arange(len(thresholds))
    mean_gains = []
    baseline_labels = []
    for threshold in thresholds:
        walk_row = summary[(summary.graph == 'walk') & (summary.metric == 'reachable_schools_mean')]
        intermodal_row = summary[(summary.graph == 'intermodal') & (summary.metric == 'reachable_schools_mean')]
        walk_value = float(walk_row[walk_row.threshold_min == threshold].value.iloc[0])
        intermodal_value = float(intermodal_row[intermodal_row.threshold_min == threshold].value.iloc[0])
        mean_gains.append(intermodal_value - walk_value)
        baseline_labels.append(f'{walk_value:.1f} → {intermodal_value:.1f}')
    gain_bars = axes[1].bar(x, mean_gains, width=0.62, color=C['iduedu'], zorder=3)
    axes[1].bar_label(gain_bars, labels=[f'+{value:.1f}' for value in mean_gains], padding=3)
    axes[1].set_xticks(
        x, [f'{threshold} min\n{label}' for threshold, label in zip(thresholds, baseline_labels)]
    )
    axes[1].set_ylabel('Additional schools per origin')
    axes[1].set_title('Mean gain from public transport', fontweight='bold')
    style_ax(axes[1], grid=True)

    class_order = ['no_effect', 'moderate', 'substantial', 'newly_reachable', 'unreachable']
    shares = origins.gain_class.value_counts(normalize=True).reindex(class_order).fillna(0) * 100
    axes[2].barh(
        np.arange(len(class_order)),
        shares.values,
        color=[GAIN_COLORS[value] for value in class_order],
        edgecolor='white',
    )
    axes[2].set_yticks(np.arange(len(class_order)), [value.replace('_', ' ').title() for value in class_order])
    axes[2].invert_yaxis()
    axes[2].set_xlabel('Origins (%)')
    axes[2].set_title('15-minute spatial effect classes', fontweight='bold')
    style_ax(axes[2], grid=True)
    save(fig, 'accessibility_gain_metrics.png')

## Map of zones where public transport changes accessibility

In [ ]:
if origins is not None:
    fig, ax = plt.subplots(figsize=(10, 8), dpi=DPI)
    plot_network(ax, walk_edges_m, pt_edges_m)
    gain_cmap = LinearSegmentedColormap.from_list(
        'accessibility_gain', ['#D9D9D9', '#FFD166', '#00A8FF', '#0057A8']
    )
    gain_values = origins_points.opportunity_gain_15.to_numpy(dtype=float)
    gain_vmax = max(1.0, float(np.nanpercentile(gain_values, 95)))
    gain_hex = projected_hexbin(
        ax, origins_points, gain_values, gridsize=120,
        reduce_C_function=np.mean, mincnt=1, cmap=gain_cmap,
        vmin=0, vmax=gain_vmax, linewidths=0, alpha=0.88, zorder=5, rasterized=True,
    )
    colorbar = fig.colorbar(gain_hex, ax=ax, shrink=0.72, pad=0.01)
    colorbar.set_label('Mean additional schools reachable within 15 min (capped at P95)')
    if schools_m is not None:
        schools_m.plot(
            ax=ax, color='black', marker='.', markersize=4, alpha=0.55,
            zorder=7, rasterized=True,
        )
    style_map(ax)
    ax.set_title(
        'Public-transport effect on 15-minute school accessibility\n'
        'Historical Saint Petersburg boundary (OSM 421007)',
        fontsize=14, fontweight='bold',
    )
    save(fig, 'accessibility_gain_map.png')

## ObjectNat isochrones and coverage

In [ ]:
isochrones = load_gdf('isochrones.parquet')
selected = load_gdf('selected_origins.parquet')
if isochrones is not None and selected is not None:
    selection = 'highest_gain' if 'highest_gain' in set(selected.selection) else selected.selection.iloc[0]
    selected_m = selected.to_crs(MAP_CRS)
    selected_m.geometry = selected_m.geometry.representative_point()
    fig, axes = plt.subplots(1, 2, figsize=(13, 6), dpi=DPI)
    for ax, graph in zip(axes, ['walk', 'intermodal']):
        plot_network(ax, walk_edges_m, pt_edges_m if graph == 'intermodal' else None)
        layer = isochrones[(isochrones.graph == graph) & (isochrones.selection == selection)].to_crs(MAP_CRS)
        if visual_boundary_m is not None:
            layer = layer.clip(visual_boundary_m)
        layer.plot(ax=ax, color=C[graph], alpha=0.28, edgecolor=C[graph], linewidth=1.8, zorder=4)
        selected_m[selected_m.selection == selection].plot(
            ax=ax, color='black', marker='*', markersize=110, zorder=8
        )
        style_map(ax)
        ax.set_title(f'{graph.capitalize()} — 30-minute isochrone', fontweight='bold')
    save(fig, 'objectnat_isochrone_comparison.png')

coverage = load_gdf('stepped_coverage.parquet')
if coverage is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=DPI)
    coverage_steps = sorted(pd.to_numeric(coverage['dist'], errors='coerce').dropna().unique())
    cmap = LinearSegmentedColormap.from_list(
        'steps', ['#E6F4FF', '#B9E0FF', '#80C7FF', '#47A8F5', '#167FCB', '#005A9C'],
        N=len(coverage_steps),
    )
    step_colors = {value: cmap(i / max(1, len(coverage_steps) - 1)) for i, value in enumerate(coverage_steps)}
    for ax, graph in zip(axes, ['walk', 'intermodal']):
        plot_network(ax, walk_edges_m, pt_edges_m if graph == 'intermodal' else None)
        layer = coverage[coverage.graph == graph].to_crs(MAP_CRS)
        if visual_boundary_m is not None:
            layer = layer.clip(visual_boundary_m)
        layer.plot(
            ax=ax, color=layer['dist'].map(step_colors), alpha=0.62,
            edgecolor='white', linewidth=0.25, zorder=3,
        )
        schools_m.plot(
            ax=ax, color='black', marker='.', markersize=3, alpha=0.5, zorder=7, rasterized=True
        )
        style_map(ax)
        ax.set_title(f'{graph.capitalize()} — school coverage, 5-min bands to 30 min', fontweight='bold')
    handles = [Patch(facecolor=step_colors[value], label=f'{value:g} min') for value in coverage_steps]
    axes[0].legend(handles=handles, loc='upper left', frameon=True, title='Nearest school')

    coverage_m = coverage.to_crs(MAP_CRS)
    if visual_boundary_m is not None:
        coverage_m = coverage_m.clip(visual_boundary_m)
    combined = combine_geometry(coverage_m[['graph', 'dist', 'geometry']])

    def distance_for_graph(row, graph):
        values = [float(dist) for name, dist in zip(row['graph'], row['dist']) if name == graph]
        return min(values) if values else np.nan

    combined['walk_dist'] = combined.apply(distance_for_graph, axis=1, graph='walk')
    combined['intermodal_dist'] = combined.apply(distance_for_graph, axis=1, graph='intermodal')
    combined['new_within_30'] = combined.walk_dist.isna() & combined.intermodal_dist.notna()
    comparison_walk = combined.walk_dist.fillna(35.0)
    comparison_intermodal = combined.intermodal_dist.fillna(35.0)
    combined['minutes_gained'] = comparison_walk - comparison_intermodal
    difference = combined[combined.minutes_gained > 0].copy()
    difference_cmap = LinearSegmentedColormap.from_list(
        'coverage_gain', ['#FFF3B0', '#FFB703', '#FB5607', '#D90429']
    )
    difference_vmax = max(5.0, float(np.nanpercentile(difference.minutes_gained, 95)))
    difference.plot(
        ax=axes[2], column='minutes_gained', cmap=difference_cmap, vmin=0, vmax=difference_vmax,
        alpha=0.94, edgecolor='white', linewidth=0.2, legend=True, zorder=3,
        legend_kwds={'label': 'Nearest-school time-band gain (min; capped at P95)', 'shrink': 0.72},
    )
    for mode, group in pt_edges_m.groupby('type'):
        group.plot(
            ax=axes[2], color=PT_COLORS.get(str(mode), C['pt']),
            linewidth=0.65, alpha=0.24, zorder=6, rasterized=True,
        )
    if visual_boundary_m is not None:
        visual_boundary_m.boundary.plot(ax=axes[2], color='#4A4A4A', linewidth=0.8, zorder=7)
    style_map(axes[2])
    axes[2].set_title('Intermodal gain — nearest-school time band', fontweight='bold')
    save(fig, 'objectnat_school_coverage.png')

## P0 — waiting-time scenarios

In [ ]:
scenario_path = RESULTS / 'scenario_summary.csv'
scenario_summary = pd.read_csv(scenario_path) if scenario_path.exists() else None
scenario_run_path = RESULTS / 'scenario_summary.json'
if scenario_summary is not None and scenario_run_path.exists():
    run_hash = json.loads(scenario_run_path.read_text(encoding='utf-8'))['experiment_hash']
    scenario_summary = scenario_summary[scenario_summary.experiment_hash.astype(str) == str(run_hash)]
if scenario_summary is not None:
    p0 = scenario_summary[scenario_summary.scenario_group.str.startswith('P0')].copy()
    selected_rows = []
    for group in ['P0_mode_wait', 'P0_route_wait']:
        part = p0[p0.scenario_group == group].sort_values('extra_wait_min')
        if not part.empty:
            selected_rows.append(part.iloc[-1])
    if selected_rows:
        n_panels = len(selected_rows)
        fig = plt.figure(figsize=(5.8 * n_panels, 6.2), dpi=DPI)
        width_ratios = []
        for index in range(n_panels):
            width_ratios.extend((1.0, 0.035))
            if index < n_panels - 1:
                width_ratios.append(0.10)
        grid = fig.add_gridspec(
            1, len(width_ratios), width_ratios=width_ratios,
            left=0.04, right=0.96, bottom=0.07, top=0.855, wspace=0.04,
        )
        axes = [fig.add_subplot(grid[0, 3 * index]) for index in range(n_panels)]
        color_axes = [fig.add_subplot(grid[0, 3 * index + 1]) for index in range(n_panels)]
        for ax, color_ax, row in zip(axes, color_axes, selected_rows):
            detail_path = RESULTS / 'scenario_origins' / f'{scenario_slug(row.scenario_id)}.parquet'
            detail = gpd.read_parquet(detail_path).to_crs(MAP_CRS)
            detail.geometry = detail.geometry.representative_point()
            if visual_boundary_m is not None:
                detail = detail.loc[detail.intersects(boundary_geometry)].copy()
            plot_network(
                ax, walk_edges_m, pt_edges_m, pt_alpha=0.26, pt_linewidth=0.60, pt_color='#9E9E9E'
            )
            loss_column = 'lost_school_opportunities_30'
            affected = detail.loc[detail[loss_column] > 0].copy()
            route_mask = pt_edges_m['type'].astype(str).eq(str(row['mode']))
            if row['scenario_group'] == 'P0_route_wait':
                route_mask &= pt_edges_m['route'].astype(str).eq(str(row['route']))
            route_edges = pt_edges_m.loc[route_mask]
            plot_highlighted_routes(ax, route_edges)
            vmax = 25.0  # shared scale across P0 panels; just above the larger panel's P95
            loss_hex = projected_hexbin(
                ax, affected, affected[loss_column], gridsize=145,
                reduce_C_function=np.mean, mincnt=1, cmap='viridis',
                vmin=0, vmax=vmax, linewidths=0, alpha=0.97, zorder=10, rasterized=True,
            )
            colorbar = fig.colorbar(loss_hex, cax=color_ax)
            colorbar.set_label('Mean schools lost per affected origin within 30 min')
            style_map(ax)
            if row['scenario_group'] == 'P0_route_wait':
                zoom_to_layers(ax, [affected, route_edges])
            ax.set_aspect('equal', adjustable='datalim')
            title_value = (
                row['mode'] if row['scenario_group'] == 'P0_mode_wait'
                else f"{row['mode']} {row['route']}"
            )
            title_position = ax.get_subplotspec().get_position(fig)
            fig.text(
                (title_position.x0 + title_position.x1) / 2, 0.93,
                f'{title_value}: +{row.extra_wait_min:g} min waiting\n'
                f"{int(row['n_origins_losing_opportunities_30'])} origins; "
                f"{int(row['total_school_opportunities_lost_30'])} opportunities lost",
                ha='center', va='center', fontweight='bold',
            )
        save(fig, 'scenario_p0_waiting_delta.png', tight=False, crop=False)

## P1 — route criticality

In [ ]:
criticality_path = RESULTS / 'route_criticality.csv'
criticality = pd.read_csv(criticality_path) if criticality_path.exists() else None
if criticality is not None and not criticality.empty:
    top = (
        criticality.groupby('mode', sort=True, group_keys=False).head(3).copy()
        .sort_values('total_school_opportunities_lost_30').reset_index(drop=True)
    )
    labels = [f'{mode} {route}' for mode, route in zip(top['mode'], top['route'])]
    fig, ax = plt.subplots(figsize=(10, 6), dpi=DPI)
    y = np.arange(len(top))
    bars_15 = ax.barh(
        y - 0.19, top.total_school_opportunities_lost_15, height=0.36,
        color=C['walk'], label='15-minute school opportunities lost',
    )
    bars_30 = ax.barh(
        y + 0.19, top.total_school_opportunities_lost_30, height=0.36,
        color=C['red'], label='30-minute school opportunities lost',
    )
    ax.set_yticks(np.arange(len(top)), labels)
    ax.set_xscale('symlog', linthresh=1_000)
    ax.bar_label(bars_30, labels=[f'{value:,.0f}' for value in top.total_school_opportunities_lost_30], padding=3)
    ax.set_xlabel('Total lost origin–school opportunities (symlog scale)')
    ax.set_title(
        'Leave-one-route-out structural criticality: top three routes per mode',
        fontsize=14, fontweight='bold',
    )
    ax.legend(frameon=True)
    style_ax(ax, grid=True)
    save(fig, 'route_criticality_ranking.png')

    top_routes_path = RESULTS / 'top_routes.json'
    if top_routes_path.exists():
        top_routes = json.loads(top_routes_path.read_text(encoding='utf-8'))['routes']
        n_routes = len(top_routes)
        fig = plt.figure(figsize=(5.5 * n_routes, 6.2), dpi=DPI)
        width_ratios = []
        for index in range(n_routes):
            width_ratios.extend((1.0, 0.035))
            if index < n_routes - 1:
                width_ratios.append(0.10)
        grid = fig.add_gridspec(
            1, len(width_ratios), width_ratios=width_ratios,
            left=0.04, right=0.96, bottom=0.07, top=0.855, wspace=0.04,
        )
        axes = [fig.add_subplot(grid[0, 3 * index]) for index in range(n_routes)]
        color_axes = [fig.add_subplot(grid[0, 3 * index + 1]) for index in range(n_routes)]
        for ax, color_ax, route_info in zip(axes, color_axes, top_routes):
            scenario_id = route_info['scenario_id']
            detail_path = RESULTS / 'scenario_origins' / f'{scenario_slug(scenario_id)}.parquet'
            detail = gpd.read_parquet(detail_path).to_crs(MAP_CRS)
            detail.geometry = detail.geometry.representative_point()
            if visual_boundary_m is not None:
                detail = detail.loc[detail.intersects(boundary_geometry)].copy()
            plot_network(
                ax, walk_edges_m, pt_edges_m, pt_alpha=0.26, pt_linewidth=0.60, pt_color='#9E9E9E'
            )
            route_edges = pt_edges_m[
                (pt_edges_m['type'].astype(str) == str(route_info['mode']))
                & (pt_edges_m['route'].astype(str) == str(route_info['route']))
            ]
            affected = detail[detail.lost_school_opportunities_30 > 0].copy()
            plot_highlighted_routes(ax, route_edges)
            vmax = max(1.0, float(np.nanpercentile(affected.lost_school_opportunities_30, 95)))
            loss_hex = projected_hexbin(
                ax, affected, affected.lost_school_opportunities_30, gridsize=145,
                reduce_C_function=np.mean, mincnt=1, cmap='viridis',
                vmin=0, vmax=vmax, linewidths=0, alpha=0.97, zorder=10, rasterized=True,
            )
            colorbar = fig.colorbar(loss_hex, cax=color_ax)
            colorbar.set_label('Mean schools lost per affected origin within 30 min')
            style_map(ax)
            zoom_to_layers(ax, [affected, route_edges])
            ax.set_aspect('equal', adjustable='datalim')
            title_position = ax.get_subplotspec().get_position(fig)
            fig.text(
                (title_position.x0 + title_position.x1) / 2, 0.93,
                f"{route_info['mode']} {route_info['route']}\n"
                f"30-min opportunities lost: {int(route_info['total_school_opportunities_lost_30']):,}",
                ha='center', va='center', fontweight='bold',
            )
        save(fig, 'scenario_p1_top_routes.png', tight=False, crop=False)